Scoring: one bootstrap run over the main-run answers. Writes `results/results.json` (flat key → value, every number
the text cites), `results/replicates.parquet` and `results/draws.parquet` (box multiplicities per replicate, for reuse).

All metrics go through `thesis.eval.scorer`: primary estimand design-weighted, w = (N_h/m_h)/n_box, Rao-Wu draws
(`bootstrap_draws`), identical draws for all models. Invalid = wrong (median log error: +inf).
Sensitivities on the same draws: invalid excluded, ±3 m with |d| ≥ 3, per_box_mean, pooled.
Consistency: repeats 1-4 from `consistency_answers.parquet`, repeat 0 from the main run. Run `consistency_check.ipynb` first.

In [18]:
import json

import numpy as np
import pandas as pd

from thesis.config import REPO_ROOT, load_config
from thesis.eval.scorer import (
    bootstrap_draws,
    bootstrap_median_error,
    bootstrap_p_value,
    holm,
    item_weights,
    log_error,
    percentile_interval,
    reliability_boundary,
    score,
    weighted_median,
)

out_dir = REPO_ROOT / "data" / "dataset_v2"
results_dir = REPO_ROOT / "results"
results_dir.mkdir(exist_ok=True)
scoring = load_config("scoring")
SIGN_BA_NULL = 0.5  # RQ1 distance: sign BA > 0.5 (Holm)
BASELINE_LOG_ERROR = 0.578  # preregistered best-constant median log error
N_PINS = 14_205


def bin_key(b):
    return f"m{-b}" if b < 0 else f"p{b}"


excluded = {entry["source_id"] for entry in scoring["excluded_sources"]}
sources = pd.read_parquet(out_dir / "selection" / "sources.parquet")[
    ["source_id", "box_id", "ecoregion", "building_stratum", "imagery_season"]
]
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")[["pin_id", "bin_id", "exact_signed_distance_m"]]
answers = pd.read_parquet(out_dir / "main_run" / "answers.parquet").merge(sources, on="source_id")
answers["inside"] = answers["bin_id"] < 0
primary = answers.loc[(answers["purpose"] == "primary") & ~answers["source_id"].isin(excluded)]
assert (primary.groupby(["task", "model"]).size() == N_PINS).all()
verification = primary.loc[primary["task"] == "verification"]
distance = primary.loc[primary["task"] == "distance"]
MODELS = sorted(primary["model"].unique())

# the draws score() and bootstrap_median_error() compute internally (same boxes, same seed), persisted for reuse
box_ecoregion = verification.groupby("box_id")["ecoregion"].first().sort_index()
draws = bootstrap_draws(box_ecoregion, int(scoring["bootstrap_replicates"]), int(scoring["bootstrap_seed"]))
pd.DataFrame(draws, columns=box_ecoregion.index).to_parquet(results_dir / "draws.parquet")

results, replicates = {"tau": scoring["tau"], "n_primary_pins": N_PINS, "baseline_log_error": BASELINE_LOG_ERROR}, {}


def record(key, point, reps):
    low, high = percentile_interval(reps)
    results.update({key: float(point), f"{key}_ci_low": low, f"{key}_ci_high": high})
    replicates[key] = np.asarray(reps)


draws.shape

(10000, 119)

In [19]:
# baseline check only; the pass condition uses the locked constant
base = distance.loc[distance["model"] == MODELS[0]]
weight = item_weights(base, scoring["estimand"]).to_numpy()
true_m = base["exact_signed_distance_m"].to_numpy()
constant_error = {c: weighted_median(log_error(np.full_like(true_m, c), true_m), weight) for c in range(81)}
best_c = min(constant_error, key=constant_error.get)
assert round(constant_error[best_c], 3) == BASELINE_LOG_ERROR, constant_error[best_c]
results["baseline_constant_m"] = best_c

# per-bin baselines with the model estimators: best constant (median log error), always "no" (accuracy)
point, _ = bootstrap_median_error(base.assign(model="constant", parsed_answer=str(best_c)), by=["bin_id"])
results.update({f"baseline_log_error_{bin_key(b)}": float(v) for (_, b), v in point.items()})
always_no = verification.loc[verification["model"] == MODELS[0]].assign(model="always_no", parsed_answer="no")
point, _ = score(always_no, by=["model"])
results["always_no_accuracy"] = float(point.loc["always_no", "accuracy"])
point, _ = score(always_no, by=["model"], estimand="pooled")
results["always_no_accuracy_pooled"] = float(point.loc["always_no", "accuracy"])
point, _ = score(always_no, by=["model", "bin_id"])
results.update({f"always_no_accuracy_{bin_key(b)}": float(v) for (_, b), v in point["accuracy"].items()})

In [20]:
# headline metrics, once primary (suffix "") and once per sensitivity
def headline(suffix, verification, distance, **kwargs):
    point, reps = score(verification, by=["model"], **kwargs)
    for m in MODELS:
        record(f"verification_ba_{m}{suffix}", point.loc[m, "balanced_accuracy"], reps[("balanced_accuracy", m)])
        record(f"verification_precision_{m}{suffix}", point.loc[m, "precision"], reps[("precision", m)])

    # RQ2: delta = acc(+3) - acc(>= +20)
    groups = np.select([verification["bin_id"] == 3, verification["bin_id"] >= 20], ["p3", "ge20"], "other")
    point, reps = score(verification.assign(group=groups), by=["model", "group"], **kwargs)
    for m in MODELS:
        record(
            f"rq2_delta_{m}{suffix}",
            point.loc[(m, "p3"), "accuracy"] - point.loc[(m, "ge20"), "accuracy"],
            reps[("accuracy", m, "p3")] - reps[("accuracy", m, "ge20")],
        )

    point, reps = score(distance, by=["model"], sign_from_distance=True, **kwargs)
    for m in MODELS:
        record(f"distance_sign_ba_{m}{suffix}", point.loc[m, "balanced_accuracy"], reps[("balanced_accuracy", m)])

    point, reps = bootstrap_median_error(distance, **kwargs)
    for m in MODELS:
        record(f"distance_log_error_{m}{suffix}", point[m], reps[m])
        results[f"distance_pass_{m}{suffix}"] = bool(np.quantile(reps[m], 0.975) < BASELINE_LOG_ERROR)


def d_ge3(items):
    return items.loc[(items["bin_id"].abs() != 3) | (items["exact_signed_distance_m"].abs() >= 3)]


headline("", verification, distance)
headline("_invalid_excluded", verification, distance, invalid_policy="exclude")
headline("_d_ge3", d_ge3(verification), d_ge3(distance))
headline("_per_box_mean", verification, distance, estimand="per_box_mean")
headline("_pooled", verification, distance, estimand="pooled")

In [21]:
# per bin (primary): accuracy, b*, median log error
point, reps = score(verification, by=["model", "bin_id"])
for m in MODELS:
    for b in scoring["primary_bins"]:
        record(f"verification_accuracy_{m}_{bin_key(b)}", point.loc[(m, b), "accuracy"], reps[("accuracy", m, b)])
    results[f"b_star_{m}"] = reliability_boundary(reps, m, scoring["tau"])["b_star"]

point, reps = bootstrap_median_error(distance, by=["bin_id"])
for (m, b), value in point.items():
    record(f"distance_log_error_{m}_{bin_key(b)}", value, reps[(m, b)])

In [22]:
# RQ2 designed/observed factors (descriptive): verification BA, distance sign BA, median log error per level
FACTORS = {"ecoregion": "ecoregion", "stratum": "building_stratum", "season": "imagery_season"}
for name, column in FACTORS.items():
    point, reps = score(verification, by=["model", column])
    for (m, level), value in point["balanced_accuracy"].items():
        record(f"verification_ba_{m}_{name}_{level}", value, reps[("balanced_accuracy", m, level)])
    point, reps = score(distance, by=["model", column], sign_from_distance=True)
    for (m, level), value in point["balanced_accuracy"].items():
        record(f"distance_sign_ba_{m}_{name}_{level}", value, reps[("balanced_accuracy", m, level)])
    point, reps = bootstrap_median_error(distance, by=[column])
    for (m, level), value in point.items():
        record(f"distance_log_error_{m}_{name}_{level}", value, reps[(m, level)])

# "yes" rate per bin, weighted point estimate (invalid = not "yes")
weight = item_weights(verification, scoring["estimand"])
yes = weight * (verification["parsed_answer"] == "yes")
rate = (
    yes.groupby([verification["model"], verification["bin_id"]]).sum()
    / weight.groupby([verification["model"], verification["bin_id"]]).sum()
)
results.update({f"verification_yes_rate_{m}_{bin_key(b)}": float(v) for (m, b), v in rate.items()})

In [23]:
# Holm over the 9 tests
p_values = {}
for m in MODELS:
    p_values[f"rq1_verification_{m}"] = bootstrap_p_value(
        replicates[f"verification_ba_{m}"], null=0.5, alternative="greater"
    )
    p_values[f"rq1_distance_{m}"] = bootstrap_p_value(
        replicates[f"distance_sign_ba_{m}"], null=SIGN_BA_NULL, alternative="greater"
    )
    p_values[f"rq2_{m}"] = bootstrap_p_value(replicates[f"rq2_delta_{m}"])
reject = holm(p_values)
results.update({f"p_{k}": v for k, v in p_values.items()} | {f"reject_{k}": v for k, v in reject.items()})

# RQ1 human comparison, outside the family: precision model - human on the same pins, pass = 95% CI contains 0
human = (
    pd.read_json(out_dir / "gold" / "human_on_roof.jsonl", lines=True)[["pin_id", "source_id", "parsed_answer"]]
    .loc[lambda f: ~f["source_id"].isin(excluded)]
    .merge(pins, on="pin_id")
    .merge(sources, on="source_id")
    .assign(model="human")
)
human["inside"] = human["bin_id"] < 0
results["n_human_pins"] = len(human)
paired = pd.concat([human, verification.loc[verification["pin_id"].isin(human["pin_id"])]])
for suffix, kwargs in {"": {}, "_pooled": {"estimand": "pooled"}}.items():
    point, reps = score(paired, by=["model"], **kwargs)
    record(f"human_precision{suffix}", point.loc["human", "precision"], reps[("precision", "human")])
    for m in MODELS:
        record(f"human_pins_precision_{m}{suffix}", point.loc[m, "precision"], reps[("precision", m)])
        key = f"precision_minus_human_{m}{suffix}"
        record(
            key,
            point.loc[m, "precision"] - point.loc["human", "precision"],
            reps[("precision", m)] - reps[("precision", "human")],
        )
        results[f"human_pass_{m}{suffix}"] = results[f"{key}_ci_low"] <= 0 <= results[f"{key}_ci_high"]
# pooled human = descriptive 44/54
assert round(results["human_precision_pooled"], 3) == 0.815, results["human_precision_pooled"]

In [24]:
# consistency over R = 5 on the human pins: repeat 0 = main run, repeats 1-4 = consistency_check.ipynb
fields = ["task", "model", "pin_id", "parsed_answer", "raw_text", "refusal_error_state"]
repeats = pd.read_parquet(out_dir / "consistency_answers.parquet").rename(columns={"provider": "model"})
meta = verification.drop_duplicates("pin_id")[
    ["pin_id", "bin_id", "exact_signed_distance_m", "box_id", "ecoregion", "inside"]
]
consistency = pd.concat(
    [
        primary.loc[primary["pin_id"].isin(human["pin_id"]), fields].assign(repeat=0),
        repeats.loc[repeats["repeat"] > 0, [*fields, "repeat"]],
    ]
).merge(meta, on="pin_id")
assert (consistency.groupby(["task", "model"]).size() == 5 * len(human)).all()

# share of unanimous pins, mean agreement with the majority answer (invalid = own answer)
answer = consistency["parsed_answer"].fillna("invalid")
per_pin = answer.groupby([consistency["task"], consistency["model"], consistency["pin_id"]]).agg(
    majority=lambda a: a.value_counts().iloc[0] / len(a), unanimous=lambda a: a.nunique() == 1
)
for (task, m), group in per_pin.groupby(level=[0, 1]):
    results[f"consistency_{task}_unanimous_{m}"] = float(group["unanimous"].mean())
    results[f"consistency_{task}_majority_agreement_{m}"] = float(group["majority"].mean())

# task 2: median per-pin SD of the distances, share of pins whose sign changes (valid answers)
value = pd.to_numeric(consistency["parsed_answer"].where(consistency["task"] == "distance"), errors="coerce")
per_pin = value.groupby([consistency["model"], consistency["pin_id"]]).agg(
    sd="std", sign_change=lambda v: (v.dropna() <= 0).nunique() > 1
)
for m, group in per_pin.groupby(level=0):
    results[f"consistency_distance_sd_median_{m}"] = float(group["sd"].median())
    results[f"consistency_distance_sign_change_{m}"] = float(group["sign_change"].mean())

# BA per repetition (sign BA for task 2) and its spread
for (task, r), group in consistency.groupby(["task", "repeat"]):
    point, _ = score(group, by=["model"], sign_from_distance=task == "distance")
    for m in MODELS:
        results[f"consistency_{task}_ba_r{r}_{m}"] = float(point.loc[m, "balanced_accuracy"])
for task in ("verification", "distance"):
    for m in MODELS:
        values = [results[f"consistency_{task}_ba_r{r}_{m}"] for r in range(5)]
        results[f"consistency_{task}_ba_spread_{m}"] = max(values) - min(values)

# empty answers per repetition
empty = (consistency["refusal_error_state"] == "invalid_output") & (consistency["raw_text"].str.strip() == "")
for (task, m, r), n in empty.groupby([consistency["task"], consistency["model"], consistency["repeat"]]).sum().items():
    results[f"consistency_{task}_empty_r{r}_{m}"] = int(n)

In [25]:
# design effect of the per-bin verification accuracy (primary bins): deff = bootstrap variance / naive p(1-p)/n,
# deff_w = Kish weighting factor of the scoring weights, deff_c = deff / deff_w, rho = (deff_c - 1) / (m_bar - 1)
weight = item_weights(verification, scoring["estimand"])  # same weights as score()
rows = []
for (m, b), group in verification.groupby(["model", "bin_id"]):
    key = f"verification_accuracy_{m}_{bin_key(b)}"
    p, n, w = results[key], len(group), weight[group.index]
    m_bar = n / group["box_id"].nunique()
    deff = replicates[key].var() / (p * (1 - p) / n)
    deff_w = n * (w**2).sum() / w.sum() ** 2
    rows.append(
        {
            "model": m,
            "bin_id": b,
            "m_bar": m_bar,
            "deff": deff,
            "deff_w": deff_w,
            "deff_c": deff / deff_w,
            "rho": (deff / deff_w - 1) / (m_bar - 1),
        }
    )
design_effect = pd.DataFrame(rows)
results.update(
    {
        f"{column}_median": float(design_effect[column].median())
        for column in ["m_bar", "deff", "deff_w", "deff_c", "rho"]
    }
)
design_effect.round(3)

,model,bin_id,m_bar,deff,deff_w,deff_c,rho
0,anthropic,-5,14.364,1.770,1.561,1.134,0.010
1,anthropic,-3,15.395,2.099,1.708,1.229,0.016
2,anthropic,3,15.395,2.120,1.708,1.241,0.017
3,anthropic,5,15.395,1.756,1.708,1.028,0.002
4,anthropic,10,15.395,1.750,1.708,1.025,0.002
5,anthropic,20,15.370,1.845,1.708,1.080,0.006
6,anthropic,40,14.807,1.247,1.724,0.723,-0.020
7,anthropic,80,13.370,2.375,1.746,1.361,0.029
8,google,-5,14.364,1.364,1.561,0.874,-0.009
9,google,-3,15.395,2.299,1.708,1.346,0.024


In [26]:
# invalid answers by kind, cost
state = primary["refusal_error_state"]
empty = (state == "invalid_output") & (primary["raw_text"].str.strip() == "")
kinds = pd.DataFrame(
    {
        "empty": empty,
        "unparseable": (state == "invalid_output") & ~empty,
        "refusal": state.notna() & (state != "invalid_output"),
    }
)
for (task, m), counts in kinds.groupby([primary["task"], primary["model"]]).sum().iterrows():
    results.update({f"{task}_{kind}_{m}": int(n) for kind, n in counts.items()})
    results[f"{task}_invalid_rate_{m}"] = float(counts.sum() / N_PINS)
results["cost_total_usd"] = float(answers["cost"].sum())
for m, group in answers.groupby("model"):
    results[f"returned_model_id_{m}"] = ", ".join(sorted(group["returned_model_id"].unique()))
    # batch answers: timestamp = collection time
    dates = pd.to_datetime(group["timestamp"], unit="s").dt.strftime("%Y-%m-%d")
    results[f"run_start_{m}"], results[f"run_end_{m}"] = dates.min(), dates.max()

pd.DataFrame(replicates).to_parquet(results_dir / "replicates.parquet")
(results_dir / "results.json").write_text(json.dumps(results, indent=1))
len(results), reject

(1082,
 {'rq1_verification_anthropic': True,
  'rq1_distance_anthropic': True,
  'rq1_verification_google': True,
  'rq1_distance_google': True,
  'rq1_verification_openai': True,
  'rq1_distance_openai': True,
  'rq2_anthropic': True,
  'rq2_google': True,
  'rq2_openai': True})